# Thử prompt trực tiếp với OpenAI và Gemini

OpenAI dùng key trực tiếp; Gemini dùng OPENROUTER_API_KEY và model ID google/... trên OpenRouter.
API key là `None` thì bỏ qua bên đó; tham số `None` không được gửi.
Mặc định thử **5 câu**, đổi `LIMIT = None` để chạy tất cả.

Dùng nguyên prompt của bạn, chỉ nối thêm dữ liệu câu hỏi. Prompt cần yêu cầu JSON thuần gồm `id`, `relevant_docs`, `relevant_articles`.
Chạy lần lượt từng câu và lưu ngay kết quả. Chạy lại cell gọi API sẽ gọi lại từ đầu và ghi đè file nhãn của bên được bật.

[OpenAI Responses](https://developers.openai.com/api/docs/guides/text) · [OpenRouter Chat Completions](https://openrouter.ai/docs/quickstart)

In [ ]:
# Chạy một lần nếu chưa cài SDK:
# %pip install -U openai requests

### 1. Cấu hình và đọc câu hỏi

In [ ]:
import json
import os
from pathlib import Path
from getpass import getpass

OPENAI_API_KEY = (os.getenv("OPENAI_API_KEY") or getpass("OpenAI API key (Enter to skip): ")).strip() or None
OPENAI_MODEL = "gpt-6-luna"
OPENAI_PARAMS = {"temperature": None, "max_output_tokens": None}

OPENROUTER_API_KEY = (os.getenv("OPENROUTER_API_KEY") or getpass("OpenRouter API key (Enter to skip): ")).strip() or None
GEMINI_MODEL = 'google/gemini-3.8-flash'
GEMINI_PARAMS = {"temperature": None, "max_tokens": None}

PROMPT = """Bạn là chuyên gia gán nhãn ground truth cho bộ kiểm thử truy hồi văn bản pháp luật Việt Nam.
Nhiệm vụ duy nhất: từ câu hỏi được cung cấp, xác định tập văn bản và Điều luật tối thiểu nhưng đủ để trực tiếp giải quyết các yêu cầu pháp lý trong câu hỏi. 
Nguyên tắc:
 Xác định đúng bản chất vấn đề pháp lý, không chọn căn cứ chỉ vì trùng từ khóa hoặc cùng chủ đề.
Ưu tiên luật chuyên ngành; chỉ bổ sung luật chung, tố tụng hoặc lĩnh vực khác khi thực sự cần.
Chỉ lấy văn bản còn hiệu lực hoặc hiệu lực một phần tại tháng 9/2026.
Mỗi yêu cầu pháp lý chính phải có ít nhất một căn cứ trực tiếp tương ứng.
Sau khi bao phủ đủ các yêu cầu, loại bỏ mọi Điều mà nếu bỏ đi vẫn có thể giải quyết đầy đủ câu hỏi.
Ưu tiên căn cứ trực tiếp, cụ thể hơn căn cứ chung; không thêm Điều định nghĩa, nguyên tắc chung hoặc chuyển tiếp nếu đã có Điều trực tiếp hơn giải quyết cùng vấn đề.
Không bịa hoặc đoán số hiệu văn bản, số Điều; nếu không đủ chắc chắn thì không đưa vào.
relevant_docs và relevant_articles phải nhất quán, không lặp văn bản hoặc cặp law_id–article.
Về một câu hỏi sẽ chia thành 3 mức khó:
- Easy: một vấn đề pháp lý trực tiếp, ít suy luận, thường 1–2 Điều.
- Normal: nhiều tình tiết hoặc 2 yêu cầu liên quan, cần kết hợp một số quy định.
- Hard: nhiều quan hệ pháp luật hoặc nhiều tầng suy luận; có thể cần phân biệt luật chuyên ngành, ngoại lệ, thời điểm áp dụng, thủ tục hoặc nhiều văn bản.
Mức difficulty không quyết định số lượng căn cứ.
Không xác định độ khó chỉ dựa vào độ dài câu hỏi hoặc số lượng Điều.
1. Đầu vào
Đầu vào là một đối tượng JSON:
{"id":1,"question":"Câu hỏi pháp luật","topic":"Chủ đề chính","relevant_docs":[],"relevant_articles":[],"difficulty":"easy|normal|hard","secondary_topics":[]}
2. Đầu ra
Chỉ trả về một đối tượng JSON:
{"id":1,"relevant_docs":["SỐ_HIỆU_VĂN_BẢN"],"relevant_articles":[{"law_id":"SỐ_HIỆU_VĂN_BẢN","article":"Điều X"}]}
Yêu cầu định dạng:
- Chỉ trả về đúng một đối tượng JSON hợp lệ theo schema đã cho, không kèm giải thích hoặc nội dung khác.
- Không thêm trường ngoài `id`, `relevant_docs`, `relevant_articles`.
- `law_id` phải là số hiệu văn bản chính thức, ví dụ `91/2015/QH13`.
Ví dụ câu dễ:
Input:
{"id":289,"question":"Thế nào là nơi cư trú của công dân?","topic":"Hành chính tư pháp","relevant_docs":[],"relevant_articles":[],"difficulty":"easy","secondary_topics":[]}
Output:
{"id":289,"relevant_docs":["68/2020/QH14"],"relevant_articles":[{"law_id":"68/2020/QH14","article":"Điều 11"}]}
Ví dụ câu trung bình:
Input:
{"id":2471,"question":"Hai vợ chồng có ký hợp đồng mua 01 căn hộ chung cư hình thành trong tương lai. Sau đó người chồng mất không có di chúc. Người vợ có được làm công chứng văn bản cam kết tài sản riêng (tiền mua căn hộ đó là tài sản riêng của chồng) được hay không? Việc lập văn cam kết tài sản riêng này nhằm mục đích đi khai nhận di sản thừa kế.","topic":"Hôn nhân, gia đình, trẻ em, bình đẳng giới","relevant_docs":[],"relevant_articles":[],"difficulty":"normal","secondary_topics":["Dân sự","Bổ trợ tư pháp","Xây dựng, nhà ở, đô thị"]}
Output:
{"id":2471,"relevant_docs":["52/2014/QH13","91/2015/QH13","46/2024/QH15"],"relevant_articles":[{"law_id":"52/2014/QH13","article":"Điều 33"},{"law_id":"52/2014/QH13","article":"Điều 43"},{"law_id":"91/2015/QH13","article":"Điều 650"},{"law_id":"46/2024/QH15","article":"Điều 42"}]}
Ví dụ câu khó:
Input:
{"id":639,"question":"1. Khi Tòa án ra quyết định áp dụng biện pháp khẩn cấp tạm thời phong tỏa tài khoản thì Thủ trưởng cơ quan thi hành án dân sự có được ra quyết định thi hành án chủ động không? hay chỉ là một quyết định phong tỏa tài khoản thôi? 2. Khi tiền giải tỏa bồi thường di dân tái định cư nằm tại Ban quản lý dự án thì Thủ trưởng cơ quan thi hành án dân sự có được ra quyết định chủ động để phong tỏa số tiền này theo quyết định áp dụng biện pháp khẩn cấp tạm thời của Tòa án không? hay là Thủ trưởng cơ quan thi hành án dân sự giao cho Chấp hành viên làm việc với Ban quản lý dự án căn cứ vào quyết định của Tòa án để yêu cầu chuyển số tiền theo quyết định của Tòa vào tài khoản của cơ quan thi hành án dân sự thôi?","topic":"Thi hành án","relevant_docs":[],"relevant_articles":[],"difficulty":"hard","secondary_topics":["Đất đai","Tố tụng và các phương thức giải quyết tranh chấp"]}
Output:
{"id":639,"relevant_docs":["106/2025/QH15","152/2026/NĐ-CP"],"relevant_articles":[{"law_id":"106/2025/QH15","article":"Điều 33"},{"law_id":"106/2025/QH15","article":"Điều 62"},{"law_id":"106/2025/QH15","article":"Điều 67"},{"law_id":"106/2025/QH15","article":"Điều 74"},{"law_id":"152/2026/NĐ-CP","article":"Điều 43"},{"law_id":"152/2026/NĐ-CP","article":"Điều 49"},{"law_id":"152/2026/NĐ-CP","article":"Điều 50"}]}
"""
LIMIT = 20  # None để chạy tất cả

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
INPUT_PATH = ROOT / "data/test/questions_with_difficulty.json"
ORIGINAL_PATH = ROOT / "data/test/questions.json"
OUTPUT_DIR = ROOT / "data/test/batch_labeling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

questions = json.loads(INPUT_PATH.read_text(encoding="utf-8-sig"))[:LIMIT]
configs = {
    "openai": {"api_key": OPENAI_API_KEY, "model": OPENAI_MODEL, "params": OPENAI_PARAMS},
    "gemini": {"api_key": OPENROUTER_API_KEY, "model": GEMINI_MODEL, "params": GEMINI_PARAMS},
}
print(f"Số câu thử: {len(questions)}")

### 2. Gọi API trực tiếp

Mỗi câu có kết quả sẽ in tiến độ và lưu vào `labels_openai.json` / `labels_gemini.json`.
Một bên lỗi thì giữ các câu đã lưu, báo lỗi và tiếp tục bên còn lại.
Chỉ bên hoàn tất mọi câu mới được đưa vào bước gộp. Tham số Gemini dùng tên của Chat Completions, ví dụ `max_tokens`.

In [ ]:
def parse_model_json(content):
    if not isinstance(content, str) or not content.strip():
        raise ValueError("Model trả về content rỗng hoặc không phải văn bản.")
    text = content.strip().removeprefix("\ufeff").strip()
    lines = text.splitlines()
    if len(lines) >= 3 and lines[0].strip().lower() in ("```json", "```") and lines[-1].strip() == "```":
        text = "\n".join(lines[1:-1]).strip()
    result = json.loads(text)
    if not isinstance(result, dict):
        raise ValueError("Cần một JSON object cho mỗi câu hỏi.")
    for field in ("relevant_docs", "relevant_articles"):
        if not isinstance(result.get(field), list):
            raise ValueError(f"Thiếu hoặc sai kiểu danh sách: {field}")
    return result


def generate_labels(provider, config, questions, prompt):
    params = {k: v for k, v in (config["params"] or {}).items() if v is not None}
    if provider == "openai":
        from openai import OpenAI
        client = OpenAI(api_key=config["api_key"], timeout=120, max_retries=0)
    else:
        import requests as http
        client = http.Session()
        client.headers.update({"Authorization": f"Bearer {config['api_key']}"})

    labels = []
    labels_path = OUTPUT_DIR / f"labels_{provider}.json"
    labels_path.write_text("[]", encoding="utf-8")
    try:
        for index, question in enumerate(questions, 1):
            payload = {k: v for k, v in question.items() if k not in ("relevant_docs", "relevant_articles")}
            text = prompt + "\n\n" + json.dumps(payload, ensure_ascii=False)
            if provider == "openai":
                response = client.responses.create(model=config["model"], input=text, **params)
                if response.status != "completed":
                    raise RuntimeError(f"ID {question['id']}: {response.status}")
                content = response.output_text
            else:
                response = client.post(
                    "https://openrouter.ai/api/v1/chat/completions",
                    json={**params, "model": config["model"], "messages": [{"role": "user", "content": text}]},
                    timeout=120,
                )
                if not response.ok:
                    raise RuntimeError(f"OpenRouter HTTP {response.status_code}: {response.text}")
                try:
                    body = response.json()
                except ValueError as error:
                    error_path = OUTPUT_DIR / f"response_error_{provider}.txt"
                    error_path.write_text(response.text, encoding="utf-8")
                    raise ValueError(f"ID {question['id']}: OpenRouter HTTP {response.status_code} trả body không phải JSON. Xem {error_path}") from error
                if body.get("error"):
                    raise RuntimeError(body["error"])
                choice = body["choices"][0]
                if choice["finish_reason"] != "stop":
                    raise RuntimeError(f"ID {question['id']}: {choice['finish_reason']}")
                content = choice["message"]["content"]
            try:
                result = parse_model_json(content)
            except ValueError as error:
                error_path = OUTPUT_DIR / f"response_error_{provider}.txt"
                error_path.write_text(content if isinstance(content, str) else repr(content), encoding="utf-8")
                raise ValueError(f"ID {question['id']}: {error}. Xem phản hồi gốc tại {error_path}") from error
            labels.append({
                "id": question["id"],
                "relevant_docs": result["relevant_docs"],
                "relevant_articles": result["relevant_articles"],
            })
            labels.sort(key=lambda row: row["id"])
            labels_path.write_text(json.dumps(labels, ensure_ascii=False, indent=2), encoding="utf-8")
            print(f"{provider}: {index}/{len(questions)} — id={question['id']}", flush=True)
    finally:
        client.close()
    return labels_path


# Kiểm tra trước khi gọi API.
for provider, config in configs.items():
    if config["api_key"] is not None:
        assert config["api_key"].strip() and config["model"].strip(), f"Điền API key và model cho {provider}."
        assert PROMPT.strip() and questions, "Điền PROMPT và kiểm tra input."
        assert not config["model"].endswith(":batch"), "Gọi trực tiếp: bỏ hậu tố :batch khỏi model."

ready_labels = {}
for provider, config in configs.items():
    if config["api_key"] is None:
        print(f"{provider}: bỏ qua (API key = None)")
        continue
    try:
        ready_labels[provider] = generate_labels(provider, config, questions, PROMPT)
    except Exception as error:
        # Giữ kết quả các câu đã chạy và tiếp tục bên còn lại.
        print(f"{provider}: lỗi: {error}")


### 3. Gộp vào questions.json

Gộp riêng kết quả của mỗi bên đã chạy xong vào file gốc:
- `questions_completed_openai.json`
- `questions_completed_gemini.json`

Chỉ ghi đè hai trường nhãn ở ID đã thử, giữ các câu còn lại và sắp theo ID. File nằm trong `OUTPUT_DIR`.

In [ ]:
def merge_with_questions(original_path, labels_path, output_path):
    original = json.loads(Path(original_path).read_text(encoding="utf-8-sig"))
    labels = json.loads(Path(labels_path).read_text(encoding="utf-8-sig"))
    updates = {row["id"]: row for row in labels}
    original_ids = {row["id"] for row in original}
    if len(updates) != len(labels) or len(original_ids) != len(original):
        raise ValueError("File có ID trùng.")
    if updates.keys() - original_ids:
        raise ValueError("Nhãn có ID không nằm trong file gốc.")
    for row in original:
        if row["id"] in updates:
            row["relevant_docs"] = updates[row["id"]]["relevant_docs"]
            row["relevant_articles"] = updates[row["id"]]["relevant_articles"]
    original.sort(key=lambda row: row["id"])
    Path(output_path).write_text(json.dumps(original, ensure_ascii=False, indent=2), encoding="utf-8")
    return original

for provider, labels_path in ready_labels.items():
    output_path = OUTPUT_DIR / f"questions_completed_{provider}.json"
    merged = merge_with_questions(ORIGINAL_PATH, labels_path, output_path)
    print(f"Đã lưu {len(merged)} câu: {output_path.name}")
if not ready_labels:
    print("Chưa có kết quả hợp lệ để gộp. Kiểm tra lỗi ở bước gọi API.")
